In [1]:
# DATASET: SOILGRID 2.0:
# Poggio, L., de Sousa, L. M., Batjes, N. H., Heuvelink, G. B. M., Kempen, B., Ribeiro, E., and Rossiter, D.: SoilGrids 2.0: producing soil information 
#    for the globe with quantified spatial uncertainty, SOIL, 7, 217–240, https://doi.org/10.5194/soil-7-217-2021, 2021.

# Soil Organic carbon to Soil Organic Matter Conversion: 
# The commonly used conversion is the Van Bemmelen factor, which assumes soil organic matter contains 58% carbon.
# SOM = SOC × 1.724

In [2]:
# Function to subset SoilGrid data for Canada and transboundary river basin
import numpy as np
from osgeo import gdal

def scale_raster_inplace(tif_path, scale_factor=0.1):
    """
    Multiply raster values by scale_factor in-place.
    Safe for very large GeoTIFFs.
    """
    ds = gdal.Open(tif_path, gdal.GA_Update)
    if ds is None:
        raise RuntimeError(f"Cannot open {tif_path}")

    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()

    bx, by = band.GetBlockSize()
    xsize, ysize = band.XSize, band.YSize

    for y in range(0, ysize, by):
        rows = min(by, ysize - y)
        for x in range(0, xsize, bx):
            cols = min(bx, xsize - x)
            data = band.ReadAsArray(x, y, cols, rows).astype("float32")

            if nodata is not None:
                mask = data != nodata
                data[mask] *= scale_factor
            else:
                data *= scale_factor

            band.WriteArray(data, x, y)

    band.FlushCache()
    ds = None

##
from osgeo import gdal
import os

gdal.UseExceptions()

IGH_PROJ = "+proj=igh +lat_0=0 +lon_0=0 +datum=WGS84 +units=m +no_defs"
SOILGRIDS_BASE = "/vsicurl/https://files.isric.org/soilgrids/latest/data/"

def download_soilgrids_roi(
    bb_igh,
    layers,
    depths,
    out_dir,
    res=250,
    keep_igh=False,
    scale_factor=0.1
):
    os.makedirs(out_dir, exist_ok=True)

    for layer in layers:
        for depth in depths:
            print(f"Processing {layer} {depth}...")

            vrt_url = f"{SOILGRIDS_BASE}{layer}/{layer}_{depth}_mean.vrt"

            out_igh = os.path.join(out_dir, f"{layer}_{depth}_igh.tif")
            out_ll = os.path.join(out_dir, f"{layer}_{depth}_wgs84.tif")

            # --- Step 1: Warp to IGH ---
            gdal.Warp(
                out_igh,
                vrt_url,
                outputBounds=bb_igh,
                dstSRS=IGH_PROJ,
                xRes=res,
                yRes=res,
                format="GTiff",
                multithread=True,
                warpOptions=["NUM_THREADS=ALL_CPUS"],
                creationOptions=[
                    "TILED=YES",
                    "COMPRESS=DEFLATE",
                    "PREDICTOR=2",
                    "BIGTIFF=YES"
                ]
            )

            # --- Step 2: Warp to WGS84 ---
            gdal.Warp(
                out_ll,
                out_igh,
                dstSRS="EPSG:4326",
                format="GTiff",
                multithread=True,
                warpOptions=["NUM_THREADS=ALL_CPUS"],
                creationOptions=[
                    "TILED=YES",
                    "COMPRESS=DEFLATE",
                    "PREDICTOR=2",
                    "BIGTIFF=YES"
                ]
            )

            # --- Step 3: SCALE FINAL RASTER (×0.1) ---
            scale_raster_inplace(out_ll, scale_factor)

            if not keep_igh:
                os.remove(out_igh)

            print(f"✓ Saved & scaled: {out_ll}")

    print("All SoilGrids layers processed successfully.")

In [ ]:
# Example bounding box (Homolosine meters)
# Canada and transboundary river basin in Homolosine meters
xmin = -16200500.000 
ymin = 4663250.0000
xmax = -7408000.000
ymax = 8361000.000
bbox = (xmin, ymin, xmax, ymax)

layers = ["clay", "sand"]
depths = ["0-5cm", "5-15cm", "15-30cm", "30-60cm", "60-100cm", "100-200cm"]

download_soilgrids_roi(
    bb_igh=bbox,
    layers=layers,
    depths=depths,
    out_dir=r"D:\Zelalem\soil\soilgrids\raw",
    res=250,
    keep_igh=False,
    scale_factor=0.1
)

Processing clay 0-5cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_0-5cm_wgs84.tif
Processing clay 5-15cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_5-15cm_wgs84.tif
Processing clay 15-30cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_15-30cm_wgs84.tif
Processing clay 30-60cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_30-60cm_wgs84.tif
Processing clay 60-100cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_60-100cm_wgs84.tif
Processing clay 100-200cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\clay_100-200cm_wgs84.tif
Processing sand 0-5cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\sand_0-5cm_wgs84.tif
Processing sand 5-15cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\sand_5-15cm_wgs84.tif
Processing sand 15-30cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\sand_15-30cm_wgs84.tif
Processing sand 30-60cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\sand_30-60cm_wgs84.tif
Processing sand 60-100cm..

In [4]:
# Example bounding box (Homolosine meters)
# Canada and transboundary river basin in Homolosine meters
xmin = -16200500.000 
ymin = 4663250.0000
xmax = -7408000.000
ymax = 8361000.000
bbox = (xmin, ymin, xmax, ymax)

layers = ["soc"]
depths = ["0-5cm", "5-15cm", "15-30cm", "30-60cm", "60-100cm", "100-200cm"]

download_soilgrids_roi(
    bb_igh=bbox,
    layers=layers,
    depths=depths,
    out_dir=r"D:\Zelalem\soil\soilgrids\raw",
    res=250,
    keep_igh=False,
    scale_factor=0.01   
)

Processing soc 0-5cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_0-5cm_wgs84.tif
Processing soc 5-15cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_5-15cm_wgs84.tif
Processing soc 15-30cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_15-30cm_wgs84.tif
Processing soc 30-60cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_30-60cm_wgs84.tif
Processing soc 60-100cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_60-100cm_wgs84.tif
Processing soc 100-200cm...
✓ Saved & scaled: D:\Zelalem\soil\soilgrids\raw\soc_100-200cm_wgs84.tif
All SoilGrids layers processed successfully.


In [19]:
# Function that combine many geotiff file into a single multiband geotiff
import glob
import re
import os
import numpy as np
import rasterio

def depth_key(path):
    match = re.search(r'_(\d+)-(\d+)cm_', path)
    return (int(match.group(1)), int(match.group(2)))

def combine_geotiffs(pattern, output_path):
    file_list = sorted(glob.glob(pattern), key=depth_key)

    if not file_list:
        print("❌ No matching GeoTIFF files found.")
        return

    # Delete existing output file if it exists (to avoid read errors)
    if os.path.exists(output_path):
        os.remove(output_path)

    src_files = [rasterio.open(fp) for fp in file_list]

    # Read all bands from all files
    all_bands = []
    for src in src_files:
        for i in range(1, src.count + 1):
            band = src.read(i)
            all_bands.append(band)

    # Stack all bands into a single array
    stacked_array = np.stack(all_bands)

    # Use metadata from the first file
    ref = src_files[0]
    meta = ref.meta.copy()
    meta.update({
        "count": stacked_array.shape[0],
        "driver": "GTiff",
        "compress": "DEFLATE"  # Apply LZW compression
    })

    # Write to output file
    with rasterio.open(output_path, "w", **meta) as dst:
        for i in range(stacked_array.shape[0]):
            dst.write(stacked_array[i], i + 1)

    print(f"✅ Combined raster saved to: {output_path}")

In [20]:
# Excute function to combine individual geotiff into a single multiband geotiff
combine_geotiffs(
    pattern=r'D:/Zelalem/soil/soilgrids/raw/*clay*.tif',
    output_path=r'D:/Zelalem/soil/soilgrids/CanTransBasin_CLAY.tif'
)
combine_geotiffs(
    pattern=r'D:/Zelalem/soil/soilgrids/raw/*sand*.tif',
    output_path=r'D:/Zelalem/soil/soilgrids/CanTransBasin_SAND.tif'
)
combine_geotiffs(
    pattern=r'D:/Zelalem/soil/soilgrids/raw/*soc*.tif',
    output_path=r'D:/Zelalem/soil/soilgrids/CanTransBasin_SOC.tif'
)

✅ Combined raster saved to: D:/Zelalem/soil/soilgrids/CanTransBasin_CLAY.tif
✅ Combined raster saved to: D:/Zelalem/soil/soilgrids/CanTransBasin_SAND.tif
✅ Combined raster saved to: D:/Zelalem/soil/soilgrids/CanTransBasin_SOC.tif


In [1]:
# Function to calculate Soil parameters for MESH interval soil layer:
import rasterio
import numpy as np
import os
import glob
import csv
from datetime import datetime

def calculate_weights(gsde_intervals, mesh_intervals):
    gsde_start = np.array([s for s, _ in gsde_intervals])
    gsde_end   = np.array([e for _, e in gsde_intervals])

    weights_all = []
    for (m_start, m_end) in mesh_intervals:
        overlap_start = np.maximum(m_start, gsde_start)
        overlap_end   = np.minimum(m_end, gsde_end)
        w = np.maximum(0, overlap_end - overlap_start)
        total = w.sum()
        if total == 0:
            w[:] = 0
        else:
            w = w / total
        weights_all.append(w)

    return np.array(weights_all, dtype=np.float32)


def save_weights_to_csv(weights_used, gsde_intervals, mesh_intervals, output_folder, project_name="soil_weights"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(output_folder, f"{project_name}_weights_{timestamp}.csv")

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Mesh Interval"] + [f"{s}-{e} m" for s, e in gsde_intervals])
        for (m_start, m_end), w in zip(mesh_intervals, weights_used):
            writer.writerow([f"{m_start}-{m_end} m"] + [float(f"{x:.6f}") for x in w])


def apply_weights_to_geotiff(input_path, output_path, mesh_intervals, weights_used):
    if os.path.exists(output_path):
        os.remove(output_path)

    with rasterio.open(input_path) as src:
        meta = src.meta.copy()
        meta.update({
            "count": len(mesh_intervals),
            "dtype": "float32",
            "compress": "DEFLATE",
            "nodata": np.nan
        })

        with rasterio.open(output_path, "w", **meta) as dst:

            for ji, window in src.block_windows():
                block = src.read(window=window).astype(np.float32)

                B, h, w = block.shape

                if src.nodata is not None:
                    block = np.where(block == src.nodata, np.nan, block)

                out_block = np.einsum("mb, bhw -> mhw", weights_used, block)

                for band_idx in range(len(mesh_intervals)):
                    dst.write(out_block[band_idx], band_idx + 1, window=window)


def process_soil_folder(folder_path, gsde_intervals, mesh_intervals, soil_properties):
    weights_used = calculate_weights(gsde_intervals, mesh_intervals)
    save_weights_to_csv(weights_used, gsde_intervals, mesh_intervals, folder_path)

    for prop in soil_properties:
        matches = glob.glob(os.path.join(folder_path, f"*{prop}*.tif"))
        if not matches:
            continue

        input_path = matches[0]
        output_path = os.path.join(
            folder_path,
            f"{os.path.splitext(os.path.basename(input_path))[0]}_mesh_weighted.tif"
        )

        apply_weights_to_geotiff(input_path, output_path, mesh_intervals, weights_used)

# ---------------------------------------------------------------------
# Example usage
gsde_intervals = [(0, 0.05), (0.05, 0.15), (0.15, 0.30),
                  (0.30, 0.60), (0.60, 1.0), (1.0, 2.0)]
mesh_intervals = [(0, 0.1), (0.1, 0.35), (0.35, 1.2), (1.2, 4.1)]
soil_properties = ["CLAY", "SAND", "SOC"]
input_dir = r"D:/Zelalem/soil/soilgrids"
process_soil_folder(input_dir, gsde_intervals, mesh_intervals, soil_properties)

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Compute zonal statistics (mean per band) for multiple rasters over subbasin polygons
and export a combined CSV keyed by COMID.

Key features:
- Validates inputs and COMID field
- Reprojects polygons to raster CRS
- Cleans/repairs geometries (drop Z, buffer(0), polygons only)
- Nodata-safe zonal stats via exactextract
- Dynamic band renaming (no hard-coded 1..N)
- Chunked processing with per-feature retry and skip-list logging
- Deterministic merges and sorted output columns
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Tuple

import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.crs import CRS
from shapely import wkb
from shapely.geometry.base import BaseGeometry

try:
    # Shapely ≥ 2.0
    from shapely.validation import make_valid as shapely_make_valid
    HAVE_MAKE_VALID = True
except Exception:
    HAVE_MAKE_VALID = False

try:
    from shapely import force_2d as shapely_force_2d
    HAVE_FORCE_2D = True
except Exception:
    HAVE_FORCE_2D = False

from exactextract import exact_extract


# ----------------------------
# Configuration (edit paths)
# ----------------------------
SHAPE_PATH = Path(r'D:/Zelalem/MERIT/Newagg/sorted_agg_MERIT_CanTrans_subbasins.shp')
SAND_PATH  = Path(r'D:/Zelalem/soil/soilgrids/CanTransBasin_SAND_mesh_weighted.tif')
CLAY_PATH  = Path(r'D:/Zelalem/soil/soilgrids/CanTransBasin_CLAY_mesh_weighted.tif')
SOC_PATH   = Path(r'D:/Zelalem/soil/soilgrids/CanTransBasin_SOC_mesh_weighted.tif')

OUTPUT_CSV       = Path(r'D:/Zelalem/soil/soilgrids/CanTrans_soilgrid_stats_soil.csv')
FAILED_IDS_CSV   = OUTPUT_CSV.with_name(OUTPUT_CSV.stem + "_FAILED_COMIDs.csv")

KEY_FIELD = "COMID"       # join key in the shapefile
CHUNK_SIZE = 500          # number of polygons per batch (tune to your machine)
MAX_CELLS_IN_MEMORY = 5_000_000  # exactextract memory hint (tune if memory is tight)

# ----------------------------
# Utilities
# ----------------------------
def check_exists(path: Path, kind: str = "file") -> None:
    if kind == "file" and not path.is_file():
        raise FileNotFoundError(f"{kind.capitalize()} not found: {path}")
    if kind == "dir" and not path.is_dir():
        raise FileNotFoundError(f"{kind.capitalize()} not found: {path}")


def load_polygons(shape_path: Path, key_field: str) -> gpd.GeoDataFrame:
    """Load polygons and validate key field and CRS."""
    print(f"[INFO] Reading polygons: {shape_path}")
    gdf = gpd.read_file(shape_path)
    if key_field not in gdf.columns:
        raise KeyError(
            f"Key field '{key_field}' not found in {shape_path}. "
            f"Available columns: {list(gdf.columns)}"
        )
    if gdf.crs is None:
        raise ValueError(f"Polygons have no CRS defined: {shape_path}")
    # Drop null/empty
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    gdf = gdf.reset_index(drop=True)
    return gdf


def reproject_to_raster(gdf: gpd.GeoDataFrame, raster_path: Path) -> Tuple[gpd.GeoDataFrame, CRS]:
    """Reproject polygons to match the raster CRS if needed."""
    with rasterio.open(raster_path) as ds:
        r_crs = ds.crs
    if r_crs is None:
        raise ValueError(f"Raster has no CRS: {raster_path}")
    if gdf.crs != r_crs:
        print(f"[INFO] Reprojecting polygons from {gdf.crs} → {r_crs}")
        gdf = gdf.to_crs(r_crs)
    return gdf, r_crs


def force_2d(geom: BaseGeometry) -> BaseGeometry:
    """Drop Z/M dimensions. Prefer shapely.force_2d if available; else WKB round-trip."""
    if geom is None:
        return None
    if HAVE_FORCE_2D:
        try:
            return shapely_force_2d(geom)
        except Exception:
            pass
    # Fallback: WKB round-trip forcing 2D output
    try:
        return wkb.loads(wkb.dumps(geom, output_dimension=2))
    except Exception:
        return geom  # last resort: return unchanged


def geometry_repair(geom: BaseGeometry) -> BaseGeometry:
    """Repair invalid geometry using buffer(0) and (if available) make_valid."""
    if geom is None:
        return None
    g = geom
    # First: try buffer(0) (works well on projected CRS)
    try:
        g2 = g.buffer(0)
        if g2.is_valid:
            return g2
        g = g2
    except Exception:
        pass
    # Second: try make_valid if present
    if HAVE_MAKE_VALID:
        try:
            g3 = shapely_make_valid(g)
            if g3.is_valid:
                return g3
        except Exception:
            pass
    # Return whatever we have
    return g


def clean_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Clean GDF geometries: drop null/empty, force 2D, polygons only, fix invalids,
    and drop anything still invalid/empty.
    """
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    # Drop Z/M
    gdf["geometry"] = gdf.geometry.apply(force_2d)

    # Keep Polygon/MultiPolygon only
    gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

    # Fix invalids
    gdf["geometry"] = gdf.geometry.apply(geometry_repair)

    # Final filter
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty & gdf.is_valid].copy()
    gdf = gdf.reset_index(drop=True)
    return gdf


def band_count(raster_path: Path) -> int:
    with rasterio.open(raster_path) as ds:
        return ds.count


def _normalize_band_columns(df: pd.DataFrame, prefix: str, bcount: int) -> pd.DataFrame:
    """
    Rename 'band_{i}_mean' -> f'{prefix}{i}'.
    Handle single-band case where column may be 'mean' (no band prefix).
    Return a DataFrame subset only containing [KEY_FIELD, prefix*].
    """
    # Single band case
    if bcount == 1 and "mean" in df.columns:
        df = df.rename(columns={"mean": f"{prefix}1"})
        keep_cols = [c for c in df.columns if c == KEY_FIELD or c.startswith(prefix)]
        return df[keep_cols].copy()

    # Multi-band: rename band_{i}_mean -> prefix{i}
    rename_map = {}
    for i in range(1, bcount + 1):
        src_col = f"band_{i}_mean"
        if src_col in df.columns:
            rename_map[src_col] = f"{prefix}{i}"
    df = df.rename(columns=rename_map)

    # Warn if some expected band columns are missing (but continue)
    missing = [f"band_{i}_mean" for i in range(1, bcount + 1) if f"band_{i}_mean" not in rename_map]
    if missing:
        print(f"[WARN] Missing expected band columns in exact_extract output: {missing}")

    keep_cols = [c for c in df.columns if c == KEY_FIELD or c.startswith(prefix)]
    return df[keep_cols].copy()


def zonal_mean_per_band(
    raster_path: Path,
    polygons: gpd.GeoDataFrame,
    key_field: str,
    prefix: str,
    chunk_size: int = CHUNK_SIZE,
    max_cells_in_memory: int = MAX_CELLS_IN_MEMORY,
) -> Tuple[pd.DataFrame, List[int]]:
    """
    Compute mean per band for each polygon using exactextract, in chunks.
    Returns:
      - DataFrame with columns [key_field, f"{prefix}1"..]
      - list of failed key values (e.g., COMIDs) that were skipped
    """
    print(f"[INFO] Zonal mean for raster: {raster_path}")
    bcount = band_count(raster_path)
    print(f"[INFO] Bands detected: {bcount}")

    opts = {
        "include_cols": [key_field],
        "output": "pandas",
        "progress": False,               # disable to avoid writer/progress edge cases
        "strategy": "feature-sequential",# safer for many polygons
        "max_cells_in_memory": max_cells_in_memory,
    }

    stat_spec = "mean"
    out_parts: List[pd.DataFrame] = []
    failed_keys: List[int] = []

    n = len(polygons)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        sub = polygons.iloc[start:end]

        try:
            df_part = exact_extract(raster_path, sub, stat_spec, **opts)
        except RuntimeError as e:
            print(f"[WARN] exact_extract failed for rows {start}:{end} ({e}). Retrying per feature...")
            # Retry per feature to isolate offenders
            for idx, row in sub.iterrows():
                comid_val = row[key_field]
                try:
                    df_one = exact_extract(raster_path, sub.loc[[idx]], stat_spec, **opts)
                    out_parts.append(df_one)
                except Exception as ee:
                    # Try extra repair once more on geometry
                    try:
                        row2 = row.copy()
                        row2.geometry = geometry_repair(force_2d(row2.geometry))
                        sub2 = gpd.GeoDataFrame([row2], columns=sub.columns, crs=sub.crs)
                        df_one2 = exact_extract(raster_path, sub2, stat_spec, **opts)
                        out_parts.append(df_one2)
                        print(f"[INFO] Succeeded after extra repair for {key_field}={comid_val}")
                    except Exception as ee2:
                        print(f"[ERROR] Skipping {key_field}={comid_val}: {ee2}")
                        failed_keys.append(comid_val)
            continue

        out_parts.append(df_part)

    if not out_parts:
        raise RuntimeError("No results produced by exact_extract.")

    df = pd.concat(out_parts, ignore_index=True)

    # Normalize band columns
    df = _normalize_band_columns(df, prefix=prefix, bcount=bcount)

    # Ensure key dtype is consistent
    df[key_field] = df[key_field].astype(polygons[key_field].dtype)

    return df, failed_keys


def merge_many_on_key(dfs: List[pd.DataFrame], key_field: str) -> pd.DataFrame:
    """Left-join dataframes on key, preserving order and avoiding duplicate column names."""
    base = dfs[0]
    for add in dfs[1:]:
        dup_cols = [c for c in add.columns if c != key_field and c in base.columns]
        if dup_cols:
            raise ValueError(f"Duplicate columns on merge: {dup_cols}")
        base = base.merge(add, on=key_field, how="left")
    return base


def sort_measure_columns(df: pd.DataFrame, key_field: str) -> pd.DataFrame:
    """Put key first; sort other columns by prefix then numeric suffix."""
    fixed_cols = [key_field]
    other_cols = [c for c in df.columns if c != key_field]

    def sort_key(c: str):
        # Split letters prefix and trailing digits
        head = c.rstrip("0123456789")
        tail = c[len(head):]
        num = int(tail) if tail.isdigit() else 0
        return (head, num)

    other_cols_sorted = sorted(other_cols, key=sort_key)
    return df[fixed_cols + other_cols_sorted].copy()


# ----------------------------
# Main
# ----------------------------
def main() -> None:
    # Validate inputs
    for p in [SHAPE_PATH, SAND_PATH, CLAY_PATH, SOC_PATH]:
        check_exists(p, "file")

    # Load & prepare polygons
    gdf = load_polygons(SHAPE_PATH, KEY_FIELD)
    gdf, _ = reproject_to_raster(gdf, SAND_PATH)
    gdf = clean_geometries(gdf)
    print(f"[INFO] Polygons after cleaning: {len(gdf)} features")

    # Compute zonal means for each raster
    sand_df, sand_failed = zonal_mean_per_band(SAND_PATH, gdf, KEY_FIELD, prefix="meshSAND")
    clay_df, clay_failed = zonal_mean_per_band(CLAY_PATH, gdf, KEY_FIELD, prefix="meshCLAY")
    soc_df,  soc_failed  = zonal_mean_per_band(SOC_PATH,  gdf, KEY_FIELD, prefix="meshSOC")

    # Merge results
    out_df = merge_many_on_key([sand_df, clay_df, soc_df], key_field=KEY_FIELD)
    out_df = sort_measure_columns(out_df, KEY_FIELD)

    # Write outputs
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"[INFO] Wrote: {OUTPUT_CSV}")

    # Log failures, if any
    failed = sorted(set(sand_failed + clay_failed + soc_failed))
    if failed:
        pd.DataFrame({KEY_FIELD: failed}).to_csv(FAILED_IDS_CSV, index=False)
        print(f"[WARN] {len(failed)} feature(s) were skipped. See: {FAILED_IDS_CSV}")
    else:
        print("[INFO] All features processed successfully.")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"[FATAL] {exc}", file=sys.stderr)
        sys.exit(1)

[INFO] Reading polygons: D:\Zelalem\MERIT\Newagg\sorted_agg_MERIT_CanTrans_subbasins.shp
[INFO] Polygons after cleaning: 77017 features
[INFO] Zonal mean for raster: D:\Zelalem\soil\soilgrids\CanTransBasin_SAND_mesh_weighted.tif
[INFO] Bands detected: 4
[INFO] Zonal mean for raster: D:\Zelalem\soil\soilgrids\CanTransBasin_CLAY_mesh_weighted.tif
[INFO] Bands detected: 4
[INFO] Zonal mean for raster: D:\Zelalem\soil\soilgrids\CanTransBasin_SOC_mesh_weighted.tif
[INFO] Bands detected: 4
[INFO] Wrote: D:\Zelalem\soil\soilgrids\CanTrans_soilgrid_stats_soil.csv
[INFO] All features processed successfully.
